Importing YOLO

In [ ]:
from ultralytics import YOLO


Counting combination for each class

In [ ]:
import pandas as pd

#read CSV
df = pd.read_csv('dental_dataset/train/_classes.csv')

#combinations for each row
label_cols = ['caries', 'cavity', 'gingivitis', 'gum_swelling', 'healthy', 'plaque']
df['combination'] = df[label_cols].apply(lambda row: '_'.join(row.astype(str)), axis=1)

#counting combinations
print(df['combination'].value_counts())
print(f"\nTotal unique combinations: {df['combination'].nunique()}")

Organizing dataset

In [ ]:
from pathlib import Path
import shutil
import pandas as pd

source_directory = "./dental_dataset"
output_directory = "./dataset_for_yolo_proper"

def organize_dataset_with_split(split_name):
    """
    Organize images into folders based on class combinations
    """
    source_path = Path(source_directory) / split_name
    output_path = Path(output_directory) / split_name 
    csv_file = source_path / "_classes.csv"
    
    #read csv file
    df = pd.read_csv(csv_file)
    
    #combination lables
    label_cols = ['caries', 'cavity', 'gingivitis', 'gum_swelling', 'healthy', 'plaque']
    df['combination'] = df[label_cols].apply(lambda row: '_'.join(row.astype(str)), axis=1)
    
    #copying images into class folders
    for idx, row in df.iterrows():
        src = source_path / row['filename']
        destination = output_path / row['combination'] / row['filename']
        
        destination.parent.mkdir(parents=True, exist_ok=True)
        
        if src.exists() and not destination.exists():
            shutil.copy(src, destination)
    
    print(f"{len(df)} images in: {split_name}")
    return df['combination'].nunique()

#organizing the dataset
train_classes = organize_dataset_with_split("train")
test_classes = organize_dataset_with_split("test")
valid_classes = organize_dataset_with_split("valid")



Configuring YAML for YOLO

In [ ]:
import yaml
import os

#full path to dataset
dataset_path = os.path.abspath("./dataset_for_yolo_proper")

#class names from train folder
train_folder = os.path.join(dataset_path, "train")
class_names = sorted(os.listdir(train_folder))

#create YAML config
yaml_config = {
    'path': dataset_path,
    'train': 'train',
    'val': 'valid',
    'test': 'test',
    'names': {i: name for i, name in enumerate(class_names)}
}

#save YAML file
with open('dental_dataset_config.yaml', 'w') as f:
    yaml.dump(yaml_config, f)

Loading the model

In [ ]:
#loading pretrained yolo classification model
yoloModel = YOLO("yolov8n-cls.pt")



Training the model

In [ ]:
yoloModel.train(data = "dental_dataset_config.yaml", 
                epochs = 20, 
                imgsz =224,
                batch = 16,
                val = True)

